# IOAI — 2026 Local Mock Transport (Colab 자동 설정판)

아래 **설정 셀을 먼저 실행**하면 공개 데이터 소스에서 데이터를 받아 이 폴더에 `train.csv`/`test.csv` 등으로 준비합니다. 이후 셀이 그대로 학습/예측하고, 만들어진 제출 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

> 런타임 메뉴 → **런타임 유형 변경 → GPU** (필요 시).

In [ ]:
# === 데이터 자동 준비 (가장 먼저 실행) ===
import os, zipfile, urllib.request
os.makedirs('data', exist_ok=True)
if not os.path.exists('data/transport.csv'):
    urllib.request.urlretrieve('https://raw.githubusercontent.com/scvcoder/ioai-colab/main/data/roai-transport.zip', '/tmp/trans.zip')
    with zipfile.ZipFile('/tmp/trans.zip') as zf: zf.extractall('data')
print('데이터 준비:', 'data/transport.csv' if os.path.exists('data/transport.csv') else '실패')
import os; print('작업 폴더:', os.getcwd()); print('내용:', sorted(os.listdir('.')))

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.cluster import KMeans, DBSCAN

In [ ]:
root_path = "data"

seed = 42

In [ ]:
df = pd.read_csv(f"{root_path}/transport.csv")
df.head()

# Subtask 1

In [ ]:
df["id"].nunique(), len(df)

In [ ]:
df["vehicle_type"].nunique()

In [ ]:
subtask11 = df["id"].nunique()
subtask12 = df["vehicle_type"].nunique()

# Subtask 2

In [ ]:
# from this we can see there are 3 citiesA
sns.scatterplot(x="latitude", y="longitude", data=df)

In [ ]:
vehicle_coords = df.groupby("id")[["latitude", "longitude"]].median().reset_index()
# only store mean coords of each vehile, since each vehicle has multiple entries

clustering = DBSCAN(eps=0.5, min_samples=3, n_jobs=-1)
vehicle_coords["city"] = clustering.fit_predict(
    vehicle_coords[["latitude", "longitude"]]
)

vehicle_coords["city"].value_counts()

In [ ]:
plt.figure(figsize=(3,3)) 
sns.scatterplot(x="latitude", y="longitude", data=vehicle_coords)

In [ ]:
city_mapping = {
    old: new for new, old in enumerate(sorted(vehicle_coords["city"].unique()))
}
vehicle_coords["city"] = vehicle_coords["city"].map(city_mapping)

subtask2 = vehicle_coords[["id", "city"]]
subtask2.head()

# Subtask 3

In [ ]:
type10 = df[df["vehicle_type"] == 10].copy()
type10["hour"] = pd.to_datetime(type10["timestamp"]).dt.hour
type10["hour"].describe()

In [ ]:
type10 = type10[(type10["hour"] == 23) | (type10["hour"] < 3)]
len(type10)

In [ ]:
kmeans = KMeans(n_clusters=3, random_state=seed, n_init=10)
type10_coords = type10[["latitude", "longitude"]].copy()
type10_coords["depot"] = kmeans.fit_predict(type10_coords)

depots = type10_coords.groupby("depot")[["latitude", "longitude"]].mean().reset_index(drop=True)

In [ ]:
subtask3 = depots.sort_values("latitude").reset_index(drop=True)
subtask3

# Submission

In [ ]:
def build_df(sid, ans1, ans2):
    if sid != 1:
        return pd.DataFrame(
            {
                "subtaskID": [sid] * len(ans1),
                "Value1": ans1.values,
                "Value2": ans2.values,
            }
        )
    else:
        return pd.DataFrame({"subtaskID": [sid], "Value1": [ans1], "Value2": [ans2]})


subtasks = [
    (1, subtask11, subtask12),
    (2, subtask2["id"], subtask2["city"]),
    (3, subtask3["latitude"], subtask3["longitude"]),
]

submission_df = pd.concat(
    [build_df(sid, ans1, ans2) for sid, ans1, ans2 in subtasks], ignore_index=True
)

In [ ]:
submission_df.head()

In [ ]:
submission_df.to_csv("submission.csv", index=False)

## 제출 파일 모으기
아래 셀을 실행하면 제출 파일이 **최상위(`/content`)로 복사**되어 왼쪽 파일 탐색기에 바로 보입니다.
그 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

In [ ]:
# === 제출 파일을 /content 로 모으기 (마지막에 실행) ===
import os, glob, shutil
TARGETS = ['submission.csv']
OUT = "/content" if os.path.isdir("/content") else os.getcwd()
found = []
for name in TARGETS:
    hits = [name] if os.path.exists(name) else glob.glob(f"**/{name}", recursive=True)
    if not hits:
        print("아직 없음(해당 셀을 먼저 실행하세요):", name); continue
    dst = os.path.join(OUT, os.path.basename(hits[0]))
    if os.path.abspath(hits[0]) != os.path.abspath(dst):
        shutil.copy2(hits[0], dst)
    found.append(dst)
print("제출 파일 저장 위치(파일 탐색기 최상위):", found)